# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a practical guide for loading, exploring, and analyzing the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. It demonstrates how to programmatically access dataset structure, metadata, and tabular contents using the Croissant schema.

### Dataset Source
The dataset is published with a Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

*Please ensure you have a recent version of the `mlcroissant` library.*

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`. We'll define the schema URL, instantiate the dataset, and print a summary.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Name: {dataset.metadata.name}")
print(f"Version: {dataset.metadata.version}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview

List all available Record Sets, Fields, and their `@id`s as defined by the Croissant schema. All entities are referenced by `@id`.

In [ ]:
# Utility: Get all record set @ids and their fields
record_sets = list(dataset._record_sets.values())

print(f"Found {len(record_sets)} Record Sets:")
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']} (name: {rs.get('name', 'N/A')})")
    record_set_ids.append(rs['@id'])
    # List fields with @id and name
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for f in fields:
        field_obj = dataset._fields.get(f['@id']) if isinstance(f, dict) else dataset._fields.get(f)
        if field_obj:
            print(f"    Field: {field_obj['@id']} (name: {field_obj.get('name')}) - dataType: {field_obj.get('dataType')}")
        else:
            print(f"    Field: {f}")
print(f"\nAvailable Record Set IDs: {record_set_ids}")

## 3. Data Extraction

Extract and load records from a chosen Record Set using its `@id`. All referencing is by `@id` only as required by the Croissant/FAIR2 standard.

In [ ]:
# For demonstration, extract all record sets into dataframes
dataframes = {}
for record_set_id in record_set_ids:
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

print(f"DataFrames created for Record Sets: {list(dataframes.keys())}\n")
# Show available columns for the first (or pick any) record set:
if dataframes:
    example_rs = list(dataframes.keys())[0]
    print(f"Columns for RecordSet {example_rs}:\n{dataframes[example_rs].columns.tolist()}")
    dataframes[example_rs].head()

## 4. Exploratory Data Analysis (EDA)

Show how to filter, normalize, and group data from one record set. All columns/fields are referenced by their `@id` as per requirements.

- Choose a numeric field by its exact `@id`.
- Filter records using a threshold.
- Normalize the numeric field.
- Optionally group by another field (by `@id`).

In [ ]:
# Example: Analyze one numeric field in the first record set if present
if dataframes and example_rs:
    df = dataframes[example_rs]
    # Identify numeric field by @id (automatically)
    # Let's search for integer/float columns
    numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_fields:
        numeric_field_id = numeric_fields[0]  # Pick first
        print(f"Using numeric field: {numeric_field_id}\n")
        # Example threshold (may need to be adapted if values are small)
        threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (count: {len(filtered_df)})\n")
        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First 5 rows with normalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        # Try to group by another field
        group_fields = [c for c in df.columns if c != numeric_field_id and df[c].nunique() > 1 and df[c].nunique() < len(df) // 2]
        if group_fields:
            group_field_id = group_fields[0]
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped.head())
        else:
            print("No appropriate group field found for grouping.")
    else:
        print("No numeric fields found in this record set for EDA demonstration.")

## 5. Visualization

Visualize distributions or relationships using `matplotlib` or `seaborn` using fields by their `@id`. Here, we demonstrate a histogram and, if possible, a group comparison.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution if exists
if dataframes and example_rs and numeric_fields:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # If group field exists, show comparison
    if group_fields:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- We demonstrated loading a Croissant-based dataset by referencing all schema and data elements by their `@id` fields only, as recommended for FAIR workflows.
- The record sets and fields were programmatically discovered and loaded.
- Example EDA and visualizations were performed using schema `@id` references, supporting robust, schema-driven analytics.

For further work, consult the Croissant documentation and explore other related record sets and field types for deeper analysis or machine learning tasks.